# Day 4: PHI Governance & De-Identification Engine
This notebook provides an interactive de-id demo comparing raw vs. masked synthetic patient dataframes.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, regexp_replace, to_date, expr
from pyspark.sql.types import StringType
import hashlib

spark = SparkSession.builder.appName("Day04_DeID").getOrCreate()

# Sample Data
data = [
    ("P-01", "123-45-6789", "MRN-101", "1980-05-15", "Patient called from 555-123-4567 regarding meds."),
    ("P-02", "987-65-4321", "MRN-202", "1992-11-01", "Follow up required. Patient SSN 987-65-4321 noted.")
]
columns = ["patient_id", "ssn", "mrn", "dob", "clinical_notes"]
df_raw = spark.createDataFrame(data, columns).withColumn("dob", to_date(col("dob")))

print("RAW DATA:")
df_raw.show(truncate=False)

# De-identification Logic
def simulate_fpe(value):
    if not value: return value
    import os
    salt = os.environ.get('EHDIP_HASH_SALT', '')
    return hashlib.sha256((value + salt).encode('utf-8')).hexdigest()[:len(value)]
fpe_udf = udf(simulate_fpe, StringType())

def deterministic_shift_days(patient_id):
    if not patient_id: return 0
    hash_val = int(hashlib.md5(patient_id.encode('utf-8')).hexdigest(), 16)
    return (hash_val % 61) - 30
shift_udf = udf(deterministic_shift_days, StringType())

def safe_harbor_redact(text_col):
    pattern = r'\b(\d{3}-\d{2}-\d{4}|\d{3}-\d{3}-\d{4})\b'
    return regexp_replace(text_col, pattern, '[REDACTED]')

df_deid = df_raw \
    .withColumn("ssn_deid", fpe_udf(col("ssn"))) \
    .withColumn("mrn_deid", fpe_udf(col("mrn"))) \
    .withColumn("shift_days", shift_udf(col("patient_id")).cast("int")) \
    .withColumn("dob_deid", expr("date_add(dob, shift_days)")) \
    .withColumn("clinical_notes_deid", safe_harbor_redact(col("clinical_notes"))) \
    .drop("shift_days")

print("DE-IDENTIFIED DATA:")
df_deid.select("patient_id", "ssn_deid", "mrn_deid", "dob_deid", "clinical_notes_deid").show(truncate=False)